# SentinelGuard - PII Detection & Anonymization

This notebook demonstrates SentinelGuard's enterprise-grade PII (Personally Identifiable Information) detection and anonymization capabilities.

**Features covered:**
- PII detection with regex patterns (built-in, no dependencies)
- Presidio-powered detection (50+ entity types, when installed)
- Multiple anonymization strategies (replace, mask, hash, redact, fake)
- Integration with prompt/output scanning pipeline
- Data leakage prevention (OWASP LLM02)

In [ ]:
import sys
sys.path.insert(0, '..')

from sentinelguard import SentinelGuard, GuardConfig, ScannerConfig

## 1. Basic PII Scanner

The built-in PII scanner detects common PII patterns in prompts.

In [ ]:
from sentinelguard.scanners.prompt import PIIScanner

scanner = PIIScanner(threshold=0.3)

# Test with various PII types
test_cases = [
    "What is machine learning?",
    "My email is john.doe@company.com and phone is 555-123-4567",
    "Process this SSN: 123-45-6789",
    "My credit card number is 4111-1111-1111-1111",
    "I live at 123 Main Street, Anytown CA 90210",
]

print("PII Detection Results")
print("=" * 50)
for text in test_cases:
    result = scanner.scan(text)
    status = 'CLEAN' if result.is_valid else 'PII FOUND'
    print(f"[{status}] {text[:60]}")
    print(f"  Score: {result.score:.2f} | Risk: {result.risk_level.value}")
    if result.details.get("pii_found"):
        print(f"  PII Types: {result.details['pii_found']}")
    print()

## 2. PII Detector (Advanced)

The `PIIDetector` class provides more detailed PII analysis with Presidio integration when available.

In [ ]:
from sentinelguard.pii import PIIDetector

detector = PIIDetector(
    language="en",
    score_threshold=0.5,
)

text = (
    "Patient John Smith (DOB: 03/15/1985) was seen on January 10, 2025. "
    "Contact him at john.smith@hospital.com or 555-987-6543. "
    "SSN: 987-65-4321. Insurance ID: BC-12345678. "
    "Address: 456 Oak Avenue, Springfield IL 62701."
)

entities = detector.detect(text)

print("Detected PII Entities")
print("=" * 50)
for entity in entities:
    print(f"  Type: {entity.entity_type}")
    print(f"  Value: {entity.text}")
    print(f"  Score: {entity.score:.2f}")
    print(f"  Position: {entity.start}-{entity.end}")
    print()

## 3. PII Anonymization

SentinelGuard supports multiple anonymization strategies.

In [ ]:
from sentinelguard.pii import PIIAnonymizer

text = "Contact John Doe at john@example.com or call 555-123-4567"

# Detect first
detector = PIIDetector(language="en")
entities = detector.detect(text)

# Strategy 1: Replace with type labels
anonymizer = PIIAnonymizer(default_strategy="replace")
result = anonymizer.anonymize(text, entities)
print("Strategy: REPLACE")
print(f"  Original:    {text}")
print(f"  Anonymized:  {result.anonymized_text}")
print()

# Strategy 2: Mask with asterisks
anonymizer = PIIAnonymizer(default_strategy="mask")
result = anonymizer.anonymize(text, entities)
print("Strategy: MASK")
print(f"  Anonymized:  {result.anonymized_text}")
print()

# Strategy 3: Hash values
anonymizer = PIIAnonymizer(default_strategy="hash")
result = anonymizer.anonymize(text, entities)
print("Strategy: HASH")
print(f"  Anonymized:  {result.anonymized_text}")
print()

# Strategy 4: Redact completely
anonymizer = PIIAnonymizer(default_strategy="redact")
result = anonymizer.anonymize(text, entities)
print("Strategy: REDACT")
print(f"  Anonymized:  {result.anonymized_text}")

## 4. Anonymize Scanner (Prompt Pipeline)

The `AnonymizeScanner` integrates PII anonymization directly into the scanning pipeline.

In [ ]:
from sentinelguard.scanners.prompt import AnonymizeScanner

scanner = AnonymizeScanner(threshold=0.3)

text = "My name is Jane Smith and my email is jane@corp.com. My SSN is 111-22-3333."

result = scanner.scan(text)
print(f"Valid: {result.is_valid}")
print(f"Score: {result.score:.2f}")
if result.sanitized_output:
    print(f"\nOriginal:    {text}")
    print(f"Anonymized:  {result.sanitized_output}")
if result.details.get("pii_found"):
    print(f"\nPII Found: {result.details['pii_found']}")

## 5. Deanonymize Scanner (Output Pipeline)

Reverse anonymization in LLM outputs using a mapping from the anonymize step.

In [ ]:
from sentinelguard.scanners.output import DeanonymizeScanner

# Simulated anonymization mapping from a previous anonymize step
mapping = {
    "<EMAIL_0>": "jane@corp.com",
    "<PERSON_0>": "Jane Smith",
    "<PHONE_0>": "555-123-4567",
}

scanner = DeanonymizeScanner(mapping=mapping)

# LLM output with anonymized tokens
llm_output = "Dear <PERSON_0>, your order has been confirmed. We sent details to <EMAIL_0>. Call <PHONE_0> for support."

result = scanner.scan(llm_output)
print(f"Anonymized Output: {llm_output}")
print(f"Restored Output:   {result.sanitized_output}")

## 6. Data Leakage Prevention (OWASP LLM02)

Prevent sensitive data from appearing in LLM outputs.

In [ ]:
from sentinelguard.scanners.output import DataLeakageScanner

scanner = DataLeakageScanner(threshold=0.5)

# Simulated LLM outputs that might leak data
outputs = [
    "The weather in San Francisco is 65F and sunny.",
    "The patient's diagnosis is hypertension and treatment plan involves daily medication.",
    "Account balance is $45,000. Account number is 1234567890.",
    "Employee ID is EMP-12345 and their salary is $95,000.",
    "The password is: P@ssw0rd123! and the API token = eyJhbGc...",
]

print("Data Leakage Prevention (OWASP LLM02)")
print("=" * 50)
for text in outputs:
    result = scanner.scan(text)
    status = 'SAFE' if result.is_valid else 'LEAK DETECTED'
    print(f"[{status}] {text[:70]}")
    if result.details.get("categories_triggered"):
        print(f"  Leaked: {result.details['categories_triggered']}")
    print()

## 7. Full PII Protection Pipeline

Complete pipeline: anonymize inputs, send to LLM, deanonymize outputs, check for leakage.

In [ ]:
# Build a PII-focused guard
guard = SentinelGuard()
guard.use("pii", on="prompt", threshold=0.3)
guard.use("secrets", on="prompt", threshold=0.3)
guard.use("data_leakage", on="output", threshold=0.5)
guard.use("sensitive", on="output", threshold=0.5)

# Test with sensitive input
user_input = "Look up the account for john@example.com, SSN 123-45-6789"
prompt_result = guard.scan_prompt(user_input)

print("PII Protection Pipeline")
print("=" * 50)
print(f"Input: {user_input}")
print(f"Input scan: {'PASS' if prompt_result.is_valid else 'BLOCKED'}")
if not prompt_result.is_valid:
    print(f"Blocked by: {prompt_result.failed_scanners}")
    print("\nRecommendation: Anonymize PII before sending to LLM")

# Simulate safe LLM response
safe_output = "The account status is active with 3 recent transactions."
output_result = guard.scan_output(safe_output)
print(f"\nSafe output: {safe_output}")
print(f"Output scan: {'PASS' if output_result.is_valid else 'BLOCKED'}")

# Simulate leaky LLM response
leaky_output = "The account for john@example.com (SSN: 123-45-6789) has balance of $50,000"
output_result = guard.scan_output(leaky_output)
print(f"\nLeaky output: {leaky_output}")
print(f"Output scan: {'PASS' if output_result.is_valid else 'BLOCKED'}")
if not output_result.is_valid:
    print(f"Blocked by: {output_result.failed_scanners}")

## Summary

SentinelGuard provides comprehensive PII protection:

| Feature | Description |
|---------|-------------|
| **Detection** | Built-in regex + optional Presidio (50+ entity types) |
| **Anonymization** | Replace, mask, hash, redact, fake strategies |
| **Input Scanning** | `pii`, `secrets`, `anonymize` scanners |
| **Output Scanning** | `data_leakage`, `sensitive`, `deanonymize` scanners |
| **OWASP Coverage** | LLM02 (Sensitive Information Disclosure) |
| **Languages** | en, es, fr, de, it, pt, nl, he |